# MLOps Pipeline: E-Commerce Customer Churn Prediction
**Nama:** rfahrur6045  
**Dataset:** E-Commerce Customer Churn (Kaggle)  
**Framework:** TensorFlow Extended (TFX) + Apache Beam

---

## 📌 Deskripsi Proyek
Proyek ini membangun machine learning pipeline end-to-end untuk memprediksi **customer churn** pada platform e-commerce menggunakan TFX. Pipeline mencakup seluruh tahap mulai dari ingest data hingga deployment model.

In [1]:
# %pip install openpyxl

## 1. Setup & Import Library

In [2]:
# Install dependensi yang diperlukan dengan versi kompatibel
# %pip install --upgrade tfx==1.14.0 tensorflow-transform==1.14.0 tensorflow==2.13.0

In [3]:
import os
import tfx
import pandas as pd
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Trainer,
    Evaluator,
    Pusher
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.proto import example_gen_pb2, pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
import tensorflow_model_analysis as tfma


def ResolverNode(instance_name=None, resolver_class=None, **channels):
    return Resolver(strategy_class=resolver_class, **channels)

LatestBlessedModelResolver = LatestBlessedModelStrategy

print(f'TFX version: {tfx.__version__}')
print(f'TensorFlow version: {tf.__version__}')

2026-05-23 13:07:38.585951: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-23 13:07:38.883168: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorf

TFX version: 1.14.0
TensorFlow version: 2.13.0


## 2. Persiapan Dataset

Dataset yang digunakan adalah **E-Commerce Customer Churn** dari Kaggle.

- **Sumber:** https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction
- **Format asli:** Excel (`.xlsx`), sheet `E Comm`
- **Jumlah data:** 5.630 baris, 20 kolom
- **Target:** Kolom `Churn` (0 = tidak churn, 1 = churn)

Karena TFX `CsvExampleGen` hanya menerima file `.csv`, file xlsx akan dikonversi terlebih dahulu ke CSV UTF-8.

In [4]:
DATA_ROOT = '/workspace/data'
os.makedirs(DATA_ROOT, exist_ok=True)

# Tampilkan semua file yang ada di folder data
print('Files in data folder:')
for f in os.listdir(DATA_ROOT):
    print(f'  - {f}')

Files in data folder:
  - E Commerce Dataset.xlsx
  - ecommerce_churn.csv


In [5]:
# Cari file xlsx secara otomatis
xlsx_files = [f for f in os.listdir(DATA_ROOT) if f.endswith('.xlsx')]

if not xlsx_files:
    raise FileNotFoundError(
        'File .xlsx tidak ditemukan di /workspace/data. '
        'Pastikan sudah download dataset dari Kaggle.'
    )

xlsx_path = os.path.join(DATA_ROOT, xlsx_files[0])
print(f'File xlsx ditemukan: {xlsx_path}')

# Cek sheet yang tersedia
xl = pd.ExcelFile(xlsx_path)
print(f'Sheet names: {xl.sheet_names}')

File xlsx ditemukan: /workspace/data/E Commerce Dataset.xlsx
Sheet names: ['Data Dict', 'E Comm']


In [6]:
# Baca sheet 'E Comm' (sheet utama dataset)
df_raw = pd.read_excel(xlsx_path, sheet_name='E Comm')

print(f'Shape raw   : {df_raw.shape}')
print(f'Columns     : {df_raw.columns.tolist()}')
print(f'\nMissing values per kolom:')
print(df_raw.isnull().sum())
df_raw.head()

Shape raw   : (5630, 20)
Columns     : ['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

Missing values per kolom:
CustomerID                       0
Churn                            0
Tenure                         264
PreferredLoginDevice             0
CityTier                         0
WarehouseToHome                251
PreferredPaymentMode             0
Gender                           0
HourSpendOnApp                 255
NumberOfDeviceRegistered         0
PreferedOrderCat                 0
SatisfactionScore                0
MaritalStatus                    0
NumberOfAddress                  0
Complain                         0
OrderAmountHikeFromlastYear    265
CouponUsed        

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [7]:
# ── Preprocessing sebelum simpan ke CSV ───────────────────────────────

df = df_raw.copy()

# 1. Hapus kolom CustomerID (tidak dipakai sebagai fitur)
if 'CustomerID' in df.columns:
    df.drop(columns=['CustomerID'], inplace=True)
    print('CustomerID dihapus.')

# 2. Isi missing value
#    - Numerik  → isi dengan median
#    - Kategorik → isi dengan modus
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

# 3. Pastikan kolom Churn bertipe int
df['Churn'] = df['Churn'].astype(int)

print(f'Shape setelah preprocessing : {df.shape}')
print(f'Missing values tersisa      : {df.isnull().sum().sum()}')
print(f'\nDistribusi Churn:')
print(df['Churn'].value_counts())
df.head()

CustomerID dihapus.
Shape setelah preprocessing : (5630, 19)
Missing values tersisa      : 0

Distribusi Churn:
0    4682
1     948
Name: Churn, dtype: int64


,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,1,9.0,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,1,9.0,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,1,0.0,Phone,1,12.0,CC,Male,3.0,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [8]:
# Simpan sebagai CSV UTF-8 — satu-satunya file di folder data
csv_path = os.path.join(DATA_ROOT, 'ecommerce_churn.csv')
df.to_csv(csv_path, index=False, encoding='utf-8')

# Hapus file xlsx agar ExampleGen tidak membaca dua file sekaligus
os.remove(xlsx_path)

print(f'CSV tersimpan  : {csv_path}')
print(f'Isi folder data: {os.listdir(DATA_ROOT)}')

CSV tersimpan  : /workspace/data/ecommerce_churn.csv
Isi folder data: ['ecommerce_churn.csv']


## 3. Inisialisasi Pipeline Paths

Bagian ini hanya menyiapkan path artefak pipeline, metadata, model serving, serta lokasi module transform dan trainer yang akan dipakai oleh orchestrator Apache Beam.

In [10]:
# Pipeline paths and module locations (restored)
PIPELINE_NAME = 'rfahrur6045-pipeline'
PIPELINE_ROOT = os.path.join('/workspace/pipelines', PIPELINE_NAME)
METADATA_PATH = os.path.join('/workspace/metadata', PIPELINE_NAME, 'metadata.db')
SERVING_MODEL_DIR = os.path.join('/workspace/serving_model', PIPELINE_NAME)
DATA_PATH = DATA_ROOT
TRANSFORM_MODULE = os.path.join('/workspace', 'modules', 'ecommerce_transform.py')
TRAINER_MODULE = os.path.join('/workspace', 'modules', 'ecommerce_trainer.py')

# ensure directories exist
for path in [PIPELINE_ROOT, os.path.dirname(METADATA_PATH), SERVING_MODEL_DIR]:
    os.makedirs(path, exist_ok=True)

print(f'Pipeline Name    : {PIPELINE_NAME}')
print(f'Pipeline Root    : {PIPELINE_ROOT}')
print(f'Data Path        : {DATA_PATH}')
print(f'Transform Module : {TRANSFORM_MODULE}')
print(f'Trainer Module   : {TRAINER_MODULE}')

Pipeline Name    : rfahrur6045-pipeline
Pipeline Root    : /workspace/pipelines/rfahrur6045-pipeline
Data Path        : /workspace/data
Transform Module : /workspace/modules/ecommerce_transform.py
Trainer Module   : /workspace/modules/ecommerce_trainer.py


## 4. ExampleGen

Komponen **ExampleGen** membaca file CSV dan mengkonversinya ke format TFRecord. Data dibagi menjadi split **train (80%)** dan **eval (20%)** menggunakan hash-based splitting.

## 5. StatisticsGen

Komponen **StatisticsGen** menghitung statistik deskriptif dari setiap split untuk keperluan validasi data dan visualisasi distribusi fitur.

## 6. SchemaGen

Komponen **SchemaGen** menghasilkan schema dari statistik yang sudah dihitung. Schema mendefinisikan tipe data, domain nilai, dan constraint untuk setiap fitur.

## 7. ExampleValidator

Komponen **ExampleValidator** memvalidasi data berdasarkan schema, mendeteksi anomali seperti nilai hilang, tipe data salah, atau nilai di luar domain.

## 8. Transform

Komponen **Transform** melakukan preprocessing dan feature engineering:
- **Fitur numerik** → Z-score normalization (`tft.scale_to_z_score`)
- **Fitur kategorik** → Vocabulary encoding (`tft.compute_and_apply_vocabulary`)

## 9. Trainer

Komponen **Trainer** melatih model Keras dengan arsitektur:
- Input numerik (z-score) + Input kategorik (embedding dim=16)
- Dense(256) → Dropout(0.3) → Dense(128) → Dropout(0.3) → Dense(64) → Dense(1, Sigmoid)
- Optimizer: Adam (lr=0.001), Loss: Binary Crossentropy
- Metrics: AUC, Accuracy, Precision, Recall

## 10. Resolver

Komponen **Resolver** mencari model terbaik yang sudah pernah di-*bless* sebelumnya sebagai baseline untuk perbandingan evaluasi.

## 11. Evaluator

Komponen **Evaluator** mengevaluasi performa model menggunakan TFMA.

**Threshold untuk model di-bless:**
- `AUC` ≥ 0.85
- `BinaryAccuracy` ≥ 0.80

## 12. Pusher

Komponen **Pusher** mendorong model yang telah lolos evaluasi ke direktori serving. Model hanya di-push jika mendapatkan *blessing* dari Evaluator.

## 13. Menjalankan Pipeline dengan Apache Beam

Setelah seluruh komponen didefinisikan di dalam `create_pipeline`, pipeline dijalankan end-to-end menggunakan **Apache Beam** sebagai orchestrator. Tidak ada eksekusi interaktif per-komponen di notebook ini.

In [11]:
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration import pipeline as tfx_pipeline


def create_pipeline(
    pipeline_name, pipeline_root, data_path,
    transform_module, training_module,
    serving_model_dir, metadata_path
):
    example_gen = CsvExampleGen(
        input_base=data_path,
        output_config=example_gen_pb2.Output(
            split_config=example_gen_pb2.SplitConfig(
                splits=[
                    example_gen_pb2.SplitConfig.Split(name='train', hash_buckets=8),
                    example_gen_pb2.SplitConfig.Split(name='eval',  hash_buckets=2),
                ]
            )
        )
    )
    statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
    schema_gen = SchemaGen(
        statistics=statistics_gen.outputs['statistics'],
        infer_feature_shape=True
    )
    example_validator = ExampleValidator(
        statistics=statistics_gen.outputs['statistics'],
        schema=schema_gen.outputs['schema']
    )
    transform = Transform(
        examples=example_gen.outputs['examples'],
        schema=schema_gen.outputs['schema'],
        module_file=transform_module
    )
    trainer = Trainer(
        module_file=training_module,
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        train_args=trainer_pb2.TrainArgs(num_steps=500),
        eval_args=trainer_pb2.EvalArgs(num_steps=100)
    )
    model_resolver = ResolverNode(
        instance_name='latest_blessed_model_resolver',
        resolver_class=LatestBlessedModelResolver,
        model=Channel(type=Model),
        model_blessing=Channel(type=ModelBlessing)
    )
    eval_config = tfma.EvalConfig(
        model_specs=[tfma.ModelSpec(label_key='Churn')],
        slicing_specs=[tfma.SlicingSpec()],
        metrics_specs=[
            tfma.MetricsSpec(
                metrics=[
                    tfma.MetricConfig(
                        class_name='BinaryAccuracy',
                        threshold=tfma.MetricThreshold(
                            value_threshold=tfma.GenericValueThreshold(lower_bound={'value': 0.80}),
                            change_threshold=tfma.GenericChangeThreshold(
                                direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                                absolute={'value': -0.01}
                            )
                        )
                    ),
                    tfma.MetricConfig(
                        class_name='AUC',
                        threshold=tfma.MetricThreshold(
                            value_threshold=tfma.GenericValueThreshold(lower_bound={'value': 0.85}),
                            change_threshold=tfma.GenericChangeThreshold(
                                direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                                absolute={'value': -0.01}
                            )
                        )
                    ),
                ]
            )
        ]
    )
    evaluator = Evaluator(
        examples=example_gen.outputs['examples'],
        model=trainer.outputs['model'],
        baseline_model=model_resolver.outputs['model'],
        eval_config=eval_config
    )
    pusher = Pusher(
        model=trainer.outputs['model'],
        model_blessing=evaluator.outputs['blessing'],
        push_destination=pusher_pb2.PushDestination(
            filesystem=pusher_pb2.PushDestination.Filesystem(
                base_directory=serving_model_dir
            )
        )
    )
    return tfx_pipeline.Pipeline(
        pipeline_name=pipeline_name,
        pipeline_root=pipeline_root,
        metadata_connection_config=(
            tfx.orchestration.metadata
              .sqlite_metadata_connection_config(metadata_path)
        ),
        components=[
            example_gen, statistics_gen, schema_gen,
            example_validator, transform, trainer,
            model_resolver, evaluator, pusher,
        ]
    )


pipeline = create_pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    data_path=os.path.abspath(DATA_PATH),
    transform_module=TRANSFORM_MODULE,
    training_module=TRAINER_MODULE,
    serving_model_dir=os.path.abspath(SERVING_MODEL_DIR),
    metadata_path=os.path.abspath(METADATA_PATH)
)

BeamDagRunner().run(pipeline)
print('Pipeline selesai dijalankan dengan Apache Beam!')

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying ecommerce_trainer.py -> build/lib
copying ecommerce_transform.py -> build/lib
installing to /tmp/tmpe38cnz6x
running install
running install_lib
copying build/lib/ecommerce_transform.py -> /tmp/tmpe38cnz6x
copying build/lib/ecommerce_trainer.py -> /tmp/tmpe38cnz6x
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpe38cnz6x/tfx_user_code_Transform-0.0+ea113ffe9d22de904d7681ea579e245af77b96fb2bf4c042c0260f59

/opt/conda/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying ecommerce_trainer.py -> build/lib
copying ecommerce_transform.py -> build/lib
installing to /tmp/tmpbsi4jxe5
running install
running install_lib
copying build/lib/ecommerce_transform.py -> /tmp/tmpbsi4jxe5
copying build/lib/ecommerce_trainer.py -> /tmp/tmpbsi4jxe5
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmpbsi4jxe5/tfx_user_code_Trainer-0.0+ea113ffe9d22de904d7681ea579e245af77b96fb2bf4c042c0260f5936450e45-py3.10.eg

/opt/conda/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing ./pipelines/rfahrur6045-pipeline/_wheels/tfx_user_code_Transform-0.0+ea113ffe9d22de904d7681ea579e245af77b96fb2bf4c042c0260f5936450e45-py3-none-any.whl


...

INFO:tensorflow:Assets written to: /workspace/pipelines/rfahrur6045-pipeline/Trainer/model/114/Format-Serving/assets


INFO:tensorflow:Assets written to: /workspace/pipelines/rfahrur6045-pipeline/Trainer/model/114/Format-Serving/assets


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Pipeline selesai dijalankan dengan Apache Beam!


## 14. Kesimpulan

Pipeline TFX untuk prediksi customer churn e-commerce telah berhasil dibuat dan dijalankan dengan Apache Beam. Seluruh 9 komponen telah berjalan dengan baik dan model telah dipush ke direktori serving pada `serving_model/rfahrur6045-pipeline` sehingga siap untuk dideploy sebagai Hugging Face Space atau TensorFlow Serving.